Url Connection & Data Cleaning

In [12]:
import io
import os
import time
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
nasa_api = os.getenv("NASA_MAP_KEY")
source = "VIIRS_NOAA20_NRT"
day_range = 3

top_10_locations = {
  "location_1": {"name": "Amazon Rainforest", "continent": "South America", "west_lon": -80.0, "south_lat": -5.0, "east_lon": -50.0, "north_lat": 5.0},
  "location_2": {"name": "California", "continent": "North America", "west_lon": -124.5, "south_lat": 32.5, "east_lon": -114.0, "north_lat": 42.0},
  "location_3": {"name": "Siberian Taiga", "continent": "Russia", "west_lon": 60.0, "south_lat": 50.0, "east_lon": 140.0, "north_lat": 70.0},
  "location_4": {"name": "New South Wales", "continent": "Austrailia", "west_lon": 141.0, "south_lat": -37.5, "east_lon": 154.0, "north_lat": -28.0},
  "location_5": {"name": "Pantanal", "continent": "Brazil", "west_lon": -62.0, "south_lat": -22.0, "east_lon": -55.0, "north_lat": -16.0},
  "location_6": {"name": "Alberta", "continent": "Canada", "west_lon": -120.0, "south_lat": 49.0, "east_lon": -110.0, "north_lat": 60.0},
  "location_7": {"name": "Mediterranean Basin", "continent": "Southern Europe", "west_lon": -10.0, "south_lat": 30.0, "east_lon": 40.0, "north_lat": 46.0},
  "location_8": {"name": "Indonesia", "continent": "Southeast Asia", "west_lon": 95.0, "south_lat": -11.0, "east_lon": 141.0, "north_lat": 6.0},
  "location_9": {"name": "Greece", "continent": "Southern Europe", "west_lon": 19.0, "south_lat": 34.0, "east_lon": 28.0, "north_lat": 42.0},
  "location_10": {"name": "Algeria", "continent": "North Africa", "west_lon": -9.0, "south_lat": 19.0, "east_lon": 12.0, "north_lat": 37.0}
}

dataframe_list = []
master_df = None

for location_key, location_data in top_10_locations.items():
  west_lon = location_data['west_lon']
  south_lat = location_data['south_lat']
  east_lon = location_data['east_lon']
  north_lat = location_data['north_lat']
  name = location_data['name']

  data_base_url = "https://firms.modaps.eosdis.nasa.gov/api/area"
  prediction_url = f"{data_base_url}/csv/{nasa_api}/{source}/{west_lon},{south_lat},{east_lon},{north_lat}/{day_range}"

  try: 
    response = requests.get(prediction_url)
    response.raise_for_status() 
    
    if "No data available" in response.text or len(response.text.strip()) == 0:
      print(f"No active fires found in {name} for this time range.")
      continue
    
    temp_df = pd.read_csv(io.StringIO(response.text))
    temp_df["location_name"] = name

    dataframe_list.append(temp_df)
    print(f"Successfully added {len(temp_df)} fire points from {name}.")

  except requests.exceptions.HTTPError as e:
    print(f"Error fetching {name}: {e}")
  
  time.sleep(0.5)

if dataframe_list:
  master_df = pd.concat(dataframe_list, ignore_index=True)
  print("\n Collection complete!")
  print(f"Total rows collected across all locations: {len(master_df)}")
else:
  print("\n No data was collected. Check your API key or coordinates.")
master_df.head()


Successfully added 64 fire points from Amazon Rainforest.
Successfully added 85 fire points from California.
Successfully added 1653 fire points from Siberian Taiga.
Successfully added 59 fire points from New South Wales.
Successfully added 947 fire points from Pantanal.
Successfully added 101 fire points from Alberta.
Successfully added 1538 fire points from Mediterranean Basin.
Successfully added 718 fire points from Indonesia.
Successfully added 80 fire points from Greece.
Successfully added 728 fire points from Algeria.

 Collection complete!
Total rows collected across all locations: 5973


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,location_name
0,-4.06486,-79.32719,327.72,0.59,0.53,2026-05-28,714,N20,VIIRS,n,2.0NRT,288.04,3.92,N,Amazon Rainforest
1,-4.06407,-79.33247,306.64,0.59,0.53,2026-05-28,714,N20,VIIRS,n,2.0NRT,288.13,2.99,N,Amazon Rainforest
2,-4.16125,-55.13569,326.52,0.47,0.64,2026-05-28,1630,N20,VIIRS,n,2.0NRT,288.10,5.50,D,Amazon Rainforest
3,-4.16001,-55.13265,329.29,0.47,0.64,2026-05-28,1630,N20,VIIRS,n,2.0NRT,289.02,4.69,D,Amazon Rainforest
4,-3.97424,-55.77607,355.46,0.51,0.66,2026-05-28,1630,N20,VIIRS,n,2.0NRT,290.71,9.91,D,Amazon Rainforest


Data Cleaning

In [13]:
master_df.isnull().values.any() # No null values
master_df.duplicated().any() # No duplicated values
master_df.isna().sum() # No na values

latitude         0
longitude        0
bright_ti4       0
scan             0
track            0
acq_date         0
acq_time         0
satellite        0
instrument       0
confidence       0
version          0
bright_ti5       0
frp              0
daynight         0
location_name    0
dtype: int64

Data Exploration

In [14]:
variance_profile = master_df.groupby('location_name')[['frp', 'bright_ti5', 'bright_ti4']].agg(['mean','std'])
print("\n")
print(variance_profile)
print("\n")
prediction_features = master_df[['frp', 'bright_ti5', 'bright_ti4', 'track', 'scan']]
print(prediction_features.head())



                          frp             ...  bright_ti4           
                         mean        std  ...        mean        std
location_name                             ...                       
Alberta              3.712178   4.881736  ...  320.305050  19.364387
Algeria              2.946058   3.736855  ...  318.792912  16.688142
Amazon Rainforest    4.994688   2.653338  ...  335.895625  13.167838
California           1.994118   2.707725  ...  309.773647  16.500383
Greece               1.448750   1.324368  ...  309.500750  14.180689
Indonesia            7.526741  14.027086  ...  329.864471  15.741331
Mediterranean Basin  3.926515   9.200367  ...  319.749740  18.102149
New South Wales      4.523729   4.681539  ...  321.417119  17.895052
Pantanal             4.343242   6.698094  ...  322.492144  16.868304
Siberian Taiga       8.415505  13.590555  ...  327.192886  19.470317

[10 rows x 6 columns]


    frp  bright_ti5  bright_ti4  track  scan
0  3.92      288.04      327.72

Creating Risk Level Column

In [15]:
def assign_risk_level(frp_value):
  if frp_value < 2.0:
    return 0
  elif frp_value < 15.0:
    return 1
  elif frp_value < 40.0:
    return 2
  else:
    return 3

master_df['risk_level'] = master_df['frp'].apply(assign_risk_level)

Prediction: Fire Radiative Power & Bright Fire Spots

In [17]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = master_df[['track', 'scan', 'bright_ti5', 'bright_ti4']]
y = master_df['risk_level']

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size = 0.8, random_state = 42)

# Printing the shapes
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}, y_train: {y_train.shape}, y_test: {y_test.shape}")

X_train: (4778, 4), X_test: (1195, 4), y_train: (4778,), y_test: (1195,)
